# Pangeo-Fish による魚類の位置推定：海底地形と潮汐を加える（仮）

**放流から再捕までのあいだ、魚はどこを移動していたのでしょうか？** このノートブックは「[Pangeo-Fish による魚類の位置推定](pangeo-fish.ipynb)」の続きです。水温と音響テレメトリーの受信記録に加えて、海底地形と潮汐の情報も使い、各時刻に魚がいた可能性のある場所を推定します。

使うのは、M. Gonse ほかの研究 "Combining acoustic telemetry with archival tagging to investigate the spatial dynamic of the understudied pollack, *Pollachius pollachius*"（*Journal of Fish Biology* 106(5): 1400–1421, 2024, [doi:10.1111/jfb.15750](https://doi.org/10.1111/jfb.15750)）のタグ `A19124` です。このアーカイバルタグ（Data Storage Tag、DST）は水温と深度を記録しています。放流と再捕（または死亡）の情報も、位置を絞り込む追加の手がかりになります。

このノートブックは、すぐに使える FRA 横浜ワークショップ用の環境で動かすことを前提にしています。カーネルは **Python (pangeo-fish)** を選んでください。準備の詳細は「[事前登録](users-getting-started.md)」のページにあります。

Pangeo@EOSC の登録が承認されたら、まず [Pangeo@EOSC にログイン](https://pangeo-eosc.vm.fedcloud.eu/hub/login)してください。ログインしたらこのページに戻り、下の起動リンクを使います。ログインしていない状態では、起動リンクが正しく動かないことがあります。

**<a href="https://pangeo-eosc.vm.fedcloud.eu/hub/spawn?profile=only-choice&amp;profile-option-only-choice--image--unlisted-choice=ghcr.io%2Fpangeo-fish%2Fpangeo-fish-notebook%3A2026.10.21-8fb0870&amp;next=%2Fhub%2Fuser-redirect%2Fgit-pull%3Frepo%3Dhttps%253A%252F%252Fgithub.com%252Fpangeo-fish%252Fpangeo-fish-fra2026%26urlpath%3Dlab%252Ftree%252Fpangeo-fish-fra2026%252Fdocs%252Fpangeo-fish-complete.ipynb%26branch%3Dmain">ここをクリックして、ワークショップの環境を起動し、このノートブックを開きます</a>**。

または、ログインしたあとで、次の URL をコピーしてブラウザのアドレスバーに貼り付けてください。

```
https://pangeo-eosc.vm.fedcloud.eu/hub/spawn?profile=only-choice&profile-option-only-choice--image--unlisted-choice=ghcr.io%2Fpangeo-fish%2Fpangeo-fish-notebook%3A2026.10.21-8fb0870&next=%2Fhub%2Fuser-redirect%2Fgit-pull%3Frepo%3Dhttps%253A%252F%252Fgithub.com%252Fpangeo-fish%252Fpangeo-fish-fra2026%26urlpath%3Dlab%252Ftree%252Fpangeo-fish-fra2026%252Fdocs%252Fpangeo-fish-complete.ipynb%26branch%3Dmain
```

**解析の流れ**

1. **解析の準備：** ツールを読み込み、解析の設定を選びます。
2. **水温の比較：** 候補となる場所ごとに、タグの観測値と海洋モデルの水温を比べます。
3. **等面積の格子を使う：** 候補となる場所を HEALPix のセルで表します。
4. **水温と合う場所を見つける：** 水温がよく一致する場所ほど大きな重みを与えます。
5. **音響テレメトリーの受信記録を加える：** 受信機での受信記録を使って場所を絞り込みます。
6. **海底地形の情報を加える：** タグが記録した深度を海底地形と比べて、場所を絞り込みます。
7. **潮汐による手がかりを加える：** タグの深度に現れる潮汐の信号を、潮汐モデルと比べます。
8. **確率分布を統合する：** 各時刻で、これらの観測をひとつにまとめます。
9. **移動を推定する：** 時刻と時刻のあいだに魚がどのくらいの範囲を移動できるかを推定します。
10. **位置と移動経路を推定する：** 観測と移動モデルを組み合わせます。
11. **結果を見る：** 地図を確認し、移動経路を比べ、動画を作ります。

観測を時間方向につなぐために、隠れマルコフモデル（Hidden Markov Model、HMM）を使います。魚の位置は直接わかりません。このモデルは、観測と移動についての仮定の両方を使って、魚がいた可能性のある場所を推定します。

## 1. 解析の準備

セルは上から順に実行してください。コードセルをクリックして **Shift+Enter** を押すと、そのセルが実行され、次のセルに移ります。計算が終わるのを待ってから次に進んでください。セルでエラーが出たら、そのあとのステップを実行する前に講師に声をかけてください。あとで問題を報告するときは、このノートブックの最後にある付録「**バグ報告用の情報（Information for bug reports）**」のセルを実行し、その出力を報告に貼り付けてください。

ワークショップの環境には、必要な Python パッケージがすでに入っています。下の準備用のセルは、変更せずにそのまま実行してください。

このセルは、ワークショップで使う計算環境を選びます。そのまま実行してください。

In [ ]:
# CPU で実行します。
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""

### 解析ツールを読み込む

In [ ]:
from pint_xarray import unit_registry as ureg
import hvplot.xarray
import xarray as xr
import sys

# sys.path.append("../")
import pangeo_fish
from pangeo_fish.xarray_update import Accessor
from pangeo_fish.helpers import multiplot_healpix, multi_map_with_synced_sliders

### 解析の設定

設定はすべてここにまとめ、使うセクションごとに分けてあります。最初は、用意された値のまま実行してください。ご自身の研究で使うときは、タグ、解析する海域、水温の許容幅、受信機の受信範囲の 4 つが特に重要な選択になります。

In [ ]:
# セクション 1 — 初期化と設定
# タグの公開 ID。
tag_name = "A19124"

# タグデータの場所：公開 URL、ローカルのパス、または s3:// のバケットのパス。
tag_root = "https://data-taos.ifremer.fr/demo/tag/tag/"
tag_storage_options = None
# 予備：同じタグデータを Pangeo@EOSC の MinIO にミラーしたもの（公開、読み取り専用）。
# data-taos.ifremer.fr につながらないときは、上の 2 行の代わりにこちらを使います。
# tag_root = "s3://todaka-pangeo-fish-demo/tag"
# tag_storage_options = {
#     "anon": True,
#     "client_kwargs": {"endpoint_url": "https://pangeo-eosc-minioapi.vm.fedcloud.eu"},
# }

# 結果を保存するフォルダー。
scratch_root = "./test"

# scratch_root の下で結果を読み書きするための接続設定。
# None ではデフォルトの設定を使います。
# scratch_root が s3:// の場合は、認証情報・プロファイルやエンドポイントの設定を指定します。
storage_options = None

# まとめて処理する時刻の数。デフォルトの値です。
# 1 = チャンクごとに 1 時刻、-1 = すべての時刻を 1 つのチャンクに。
chunk_time = 1

# セクション 2 — 参照モデルの海洋データとアーカイバルタグ（DST）の観測を比べる
# Copernicus Marine の海洋モデルを選びます。有効にする代入は 1 つだけにしてください。
# どちらも使えます。最初は IBI を使ってください。

# IBI 再解析 — 領域モデル、約 3 km、日ごと（デフォルト）
# https://data.marine.copernicus.eu/product/IBI_MULTIYEAR_PHY_005_002
copernicus_model_name = "cmems_mod_ibi_phy-temp_my_0.027deg_P1D-m"

# GLORYS12 再解析 — 全球モデル、約 9 km、日ごと
# https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030
# copernicus_model_name = "cmems_mod_glo_phy_my_0.083deg_P1D-m"

# 選んだモデルを Copernicus Marine から読み込みます。
ref_url = None  # デフォルトの値

# モデルを読み込む範囲（緯度・経度）。セクション 3 と 9 でも使います。
bbox = {"latitude": [46, 51.5], "longitude": [-7, 0]}
# 魚が記録された深度に対して、モデルの海底が浅すぎるセルを除外します。
# 同じモデルの時間区間でのタグの最大深度にこの割合を掛けた値よりも
# 海底が浅いセルは除外されます。
# 範囲：0 以上。実用的な値は 0 から 1 のあいだです。
#   0    = フィルターなし：すべてのセルを残す（デフォルトの値）
#   0.8  = 海底がタグの最大深度の 80% 以上の深さであること
#          （モデルの海底地形の誤差をある程度許容する）
#   1    = 海底がタグの最大深度と同じかそれより深いこと
# 値が大きいほど多くのセルを除外します（より厳しい）。1 より大きいと、
# 海底が魚よりわずかに深いだけのセルも除外されます。
relative_depth_threshold = 0

# セクション 3 — HEALPix への再格子化
# 解析には HEALPix のセルを使います。セクション 3〜8 ではこの設定のままにします。
dims = ["cells"]

# HEALPix の解像度（おおよその相当ピクセルサイズ）
# refinement level が 1 つ上がるごとに、セルの面積は 1/4 になります。
# セルの総数 = 12 * 4**refinement_level
# 相当サイズ = 半径 6371 km の球としての地球上での sqrt(セルの面積)。
# 幾何学的な辺の長さそのものではありません。
#
# refinement_level = 9  # 約 12.7 km
refinement_level = 10  # 約 6.4 km
# refinement_level = 12  # 約 1.6 km
#
# refinement level を上げると空間解像度は細かくなりますが、
# セルの数と計算コストが増えます。

# セクション 3 の HEALPix への再格子化で使う地球の形。格子と一緒に保存され、
# 地図の描画と移動の探索はそこから読み込みます。ここでは "sphere" のままにします。
ellipsoid = "sphere"
# ellipsoid = "WGS84"  # 必要な精度に対して地球の扁平率が無視できないときに推奨。

# HEALPix 格子での陸と海の区別（セクション 3）。モデルの海洋マスク
# （1 = 海、0 = 陸）を HEALPix の各セルに内挿します。海岸の近くでは
# 0 から 1 のあいだの値になり、おおよそセルのまわりの海の割合を表します。
# この値が mask_threshold 以上のセルを海として残します。それ以外の
# セルは陸として扱います。データは入らず、魚はそこにいられません。
# 範囲：0 から 1。
#   0    = 少しでも海に接しているセルをすべて残す（海岸をもっとも広くとる）
#   0.25 = およそ 4 分の 1 以上が海のセルを残す（ここで使う値：
#          湾や河口を残し、モデルの海岸線よりわずかに陸側まで含む）
#   0.5  = およそ半分以上が海のセルを残す
#          （regrid_dataset_hpresample のデフォルト。モデルの海岸線に沿う）
#   1    = 海の点に囲まれたセルだけを残す
# 値が小さいほど海岸のセルを多く残し、値が大きいほど陸が残りにくくなります。
mask_threshold = 0.25

# セクション 4 — 水温にもとづく観測確率（emission probability）
# 水温の許容幅（°C）：値が大きいほど、タグとモデルの水温の差を
# より広く許容します（ガウス分布の標準偏差）。
differences_std = 0.75
# 既知の放流地点と再捕（または死亡）地点のまわりの広がり：ガウス分布の
# 標準偏差を、地球上の角度（ラジアン）で表したもの。地球の半径を掛けると
# 距離になります：1e-4 rad x 6371 km ≈ 0.64 km。
# これは HEALPix のセル 1 つ（レベル 10 で約 6.4 km）より小さいので、位置は
# 実質的に放流（または再捕）地点のセルに固定されます。
# 値が小さいほど空間的な制約が強くなります。デフォルトの値のままにしてください。
initial_std = 1e-4
recapture_std = 1e-4

# セクション 5 — 音響テレメトリーの受信による確率
# 各受信機のまわりで想定する最大受信距離。
receiver_buffer_m = 2000

# セクション 7 — 潮汐による手がかり（Complete demo のみ）
# "legacy"：緯度・経度の格子で確率分布を計算し、再格子化します（デフォルト）。
# "healpix"：セルごとの調和定数（仮）と pyfes を使います。pangeo-fish[tide] のインストールが必要です。
tide_method = "legacy"
# "tpxo9"：Ifremer が用意したデモ用データセット。"fes2022b"：AVISO の海洋潮汐（潮位）。
tide_model = "tpxo9"
# None では公開されている TPXO9 のファイルを使います。FES2022b の元ファイルを使うときは、ローカルの .nc のパスを指定します。
tide_model_source = None
# HEALPix 上の FES2022b では、元ファイルの代わりに計算済みの Zarr を指定します。
tide_constants_store = None  # s3://bucket/path.zarr またはローカルの Zarr のパス
# S3 互換のエンドポイント。None では環境の S3 設定を使います。
tide_s3_endpoint = None
# 認証情報は、カーネルの AWS 設定（AWS_PROFILE など）から読み込まれます。
# None では保存されている FES2022b の分潮（仮）をすべて使います。リストを指定すると、元ファイルと Zarr のどちらでも同じ分潮が選ばれます。
tide_constituents = None
# "healpix" 用：モデルの誤差（m）と、1 日あたりに必要な有効な 1 時間ごとのサンプル数の下限。
tide_sigma_m = 2.0
tide_min_samples = 6

# セクション 9 — 移動のパラメータの最適化
# 移動距離を角度に変換するための地球の半径は、HEALPix 格子に保存された
# 楕円体から求めます（"sphere" の場合は約 6371 km）。
# 移動のパラメータの探索上限を決めるための速度の目安。
# ありうる移動のひとつひとつに対する厳密な速度制限ではありません。
maximum_speed_km_per_day = 60
# 推定のときに試す sigma の値の上限に掛ける倍率。
# 1 では maximum_speed とモデルの時間間隔で決まる上限をそのまま使い、
# 2 ではその上限を 2 倍にします。推定される移動そのものを直接何倍かにするわけではありません。
# デフォルトでは 1 のままにします。動画のコマの間引きとは関係ありません。
adjustment_factor = 1
# FFTConv で使う、カーネルの半径（sigma の何倍か）。
truncate = 5  # デフォルトの値
# パラメータ探索の数値精度。デフォルトの値です。
tolerance = 1e-6

# セクション 8 — 状態確率と移動経路
# 移動経路の要約：各時刻の平均の位置と、もっとも確率の高いセル。
# セクション 9 の描画でも使います。
track_modes = ["viterbi_healpix", "mode", "mean"]
# ここで指定できるのは "speed"（km/h）と、1 つ前の点からの "distance"（km）です。
# ほかの量はあとから計算できます。セクション 8 の補足を参照してください。
additional_track_quantities = ["speed", "distance"]
# sigma を手で指定する比較では、平均と最頻値の移動経路を使います。ビタビは
# 近傍が広いと計算に非常に時間がかかることがあります。上の推定したモデルとは別の設定です。
comparison_track_modes = ["mode", "mean"]

# セクション 9 — 可視化と動画の作成
# 動画のフレームとフレームのあいだのデータの時刻数
# 1 = すべての時刻を含める
# 3 = 3 時刻ごとに含める
# 現在の動画作成ツールでは最後の時刻が含まれません。セクション 9 を参照してください。
time_step = 1

In [ ]:
# Papermill がパラメータを上書きしたあとで、それに依存する値を計算します。
receiver_buffer = ureg.Quantity(receiver_buffer_m, "m")
maximum_speed = ureg.Quantity(maximum_speed_km_per_day, "km / day")

if tide_method not in {"legacy", "healpix"}:
    raise ValueError('tide_method must be "legacy" or "healpix"')
if tide_model not in {"tpxo9", "fes2022b"}:
    raise ValueError('tide_model must be "tpxo9" or "fes2022b"')
if tide_constants_store and (tide_model != "fes2022b" or tide_model_source):
    raise ValueError("Select FES2022b and specify only one tide source")
if tide_constituents is not None and (
    tide_model != "fes2022b" or not tide_constituents
):
    raise ValueError("tide_constituents must be a nonempty list for FES2022b")
if tide_model == "fes2022b":
    if tide_method != "healpix" or ellipsoid != "WGS84":
        raise ValueError(
            'FES2022b requires tide_method="healpix" and ellipsoid="WGS84"'
        )
    if not tide_constants_store:
        from pathlib import Path

        if not tide_model_source or not Path(tide_model_source).expanduser().is_file():
            raise ValueError(
                "Set tide_model_source to your FES2022b NetCDF or set tide_constants_store"
            )
if tide_method == "healpix":
    from importlib.util import find_spec

    if find_spec("pyfes") is None:
        raise ImportError('Install pangeo-fish[tide] to run tide_method="healpix"')

In [ ]:
# 解析結果を保存するルートディレクトリを決めます。
target_root = f"{scratch_root}/{tag_name}"

# 最適化で使うデフォルトのチャンクサイズを決めます。
default_chunk = {"time": chunk_time, "lat": -1, "lon": -1}
default_chunk_dims = {"time": chunk_time}
default_chunk_dims.update({d: -1 for d in dims})

### 計算の準備

このセルを実行すると Dask が起動し、使える計算資源に計算を分担させます。ここで変更する設定はありません。

In [ ]:
# 分散計算のためのローカルクラスターを準備します。
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

### タグの観測データを読み込む

選んだ魚の水温・深度・イベント（放流や再捕など）の情報を読み込みます。

In [ ]:
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(
    tag_root=tag_root, tag_name=tag_name, storage_options=tag_storage_options
)
tag

In [ ]:
# 旧形式で取り込んだタグは、変数名が Depth/Temp で、深度が負の値になっています。
# 公開されている A19124 は、すでに temperature と正の値の pressure になっています。
if {"Depth", "Temp"}.issubset(tag["dst"].data_vars):
    dst = tag["dst"].to_dataset().rename({"Depth": "pressure", "Temp": "temperature"})
    dst["pressure"] = -dst[
        "pressure"
    ]  # 深度だけ符号を反転します
    tag["dst"] = dst
    tag
    tag_log = tag_log.rename({"Depth": "pressure"})
    tag_log = tag_log.rename({"Temp": "temperature"})
    tag_log["pressure"] = -tag_log["pressure"]

位置を推定する前に、記録された水温と深度の時系列をプロットして、観測データを確認します。

In [ ]:
from pangeo_fish.helpers import plot_tag

plot = plot_tag(
    tag=tag,
    tag_log=tag_log,
    # 必要ならプロットをそのまま保存できます
    save_html=True,
    storage_options=storage_options,
    target_root=target_root,
)
plot

赤い点は、タグが生きた魚に付いていたとみなす期間の観測で、解析に使われます。この期間が放流と再捕（または死亡）の情報と合っているかを確認し、記録に欠けている部分がないかも見てください。

### Copernicus Marine にサインインする

次のセルを実行してください。求められたら、Copernicus Marine のユーザー名とパスワードを入力します。ログイン情報は保存され、次回以降のセッションでも使われます。すでに保存されている場合は、もう一度聞かれることはありません。

サインインできないときは、講師に声をかけるか、[Copernicus Marine の認証情報ガイド](https://help.marine.copernicus.eu/en/articles/8185007-copernicus-marine-toolbox-credentials-configuration)を参照してください。

In [ ]:
from pathlib import Path

import copernicusmarine

credentials_file = Path.home() / ".copernicusmarine" / ".copernicusmarine-credentials"

if credentials_file.is_file():
    print("Using saved Copernicus Marine credentials.")
else:
    copernicusmarine.login()  # 必要なときにユーザー名とパスワードを尋ねます。

## 2. タグと海洋モデルの水温を比べる

### 解析する海域の海水温を読み込む

ここでは [IBI 再解析](https://data.marine.copernicus.eu/product/IBI_MULTIYEAR_PHY_005_002/services)を使います。この海域をカバーする領域海洋モデルで、日ごとのデータ、解像度は約 3 km です。次のセルで、選んだ海域とタグの装着期間のデータを読み込みます。

[GLORYS12](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/services) は全球モデルの選択肢で、日ごとのデータ、解像度は約 9 km です。モデルを切り替えるには、上の設定セルで、いま有効な `copernicus_model_name` の行の先頭に `#` を付け、もう一方の行の先頭の `#` を外します。最初は IBI で実行してください。

In [ ]:
from pangeo_fish.helpers import load_model, compute_diff

reference_model = load_model(
    uri=ref_url,
    tag_log=tag_log,
    time_slice=time_slice,
    bbox=(bbox | {"max_depth": tag_log["pressure"].max()}),
    chunk_time=chunk_time,
    remote_options={},
    copernicus_model_name=copernicus_model_name,
)

### 候補となる場所で水温を比べる

候補となる場所ごとに、タグの水温と、観測された深度でのモデルの水温を比べます。観測はモデルの時間間隔ごとにまとめられます。差が 0 に近いほど水温がよく一致していることを示しますが、似た水温の場所が複数あることもあります。

In [ ]:
%%time
diff = compute_diff(
    reference_model=reference_model,
    tag_log=tag_log,
    relative_depth_threshold=relative_depth_threshold,
    chunk_time=chunk_time,
    # 干出するセル（仮）（dry cells：干潮時に水柱 H0 + XE <= 0 となるセル。1 時間ごとの
    # North-West Shelf モデルだけで生じます）は、デフォルトで除外されます。ほかの選択肢：
    # min_water_depth="auto",  # デフォルト：干出するセルがあれば除外する
    # min_water_depth=0.5,  # 水深が 0.5 m 未満のセルも除外する
    # min_water_depth=None,  # 干出するセルを残す（以前の動作）
)[0]
diff = diff.compute()
diff

### データがそろっているかを時間ごとに確認する

このプロットは、各時刻でデータのある空間セルの数を数えたものです。欠損値（Python では `NaN` と表されます）は数えません。HMM を実行する前に、この確認が大切です。

急に減っているところや、途切れているところがないかを見てください。数が多少変動するのは問題ありませんが、急に減っている場合はデータの欠損や処理の問題かもしれません。**数が 0 になっている時刻は、その時刻に使える空間データがまったくないことを意味します。先に進む前に、講師と一緒に原因を調べてください。**

水温の差が 0 というのは完全に一致しているということなので、`.count()` ではデータのあるセルとして数えられます。

In [ ]:
%%time
diff["diff"].count(["lat", "lon"]).rename("Valid grid cells").plot()

In [ ]:
%%time
diff.to_zarr(
    f"{target_root}/diff.zarr", mode="w", storage_options=storage_options, zarr_format=2
)

## 3. 候補となる場所を等面積の格子で表す

水温の比較結果を、**HEALPix** と呼ばれる格子に移します。HEALPix は等面積の離散全球格子システム（DGGS）で、どのセルも地球表面上で同じ面積を持ちます。階層構造になっているため、細かい格子と粗い格子を選べます。このワークショップでは refinement level 10 を使い、セルの大きさは約 **6.4 km** です。

このセルの大きさは、推定した魚の位置の精度ではありません。位置の不確かさは、観測、海洋モデル、移動の仮定にも左右されます。また、等面積だからといって、セルの形がすべて同じになるわけでも、空間のゆがみがすべてなくなるわけでもありません。詳しくは [HEALPix のドキュメント](https://healpix.sourceforge.io/)を参照してください。

In [ ]:
from pangeo_fish.helpers import (
    open_diff_dataset,
    regrid_dataset_hpresample,
)

In [ ]:
# 前のステップのデータセットを開きます（ノートブックをここから再開するときだけ必要です）
diff = open_diff_dataset(target_root=target_root, storage_options=storage_options)
diff

In [ ]:
reshaped, resampling_info = regrid_dataset_hpresample(
    ds=diff,
    refinement_level=refinement_level,
    ellipsoid=ellipsoid,
    dims=dims,
    mask_threshold=mask_threshold,
)
reshaped

In [ ]:
resampling_info["bbox"] = bbox
import json

# 再格子化のメタデータと地理的な範囲を、結果と一緒に保存します。
with open(f"{target_root}/resampling_info.json", "w") as f:
    json.dump(resampling_info, f)

with open(f"{target_root}/resampling_info.json") as f:
    resampling_info = json.load(f)
resampling_info

格子を変えたので、データがそろっているかをもう一度確認します。セルの大きさや範囲が変わったため、セルの数は前と異なることがあります。急に減っているところがないかを見て、数が 0 の時刻があれば、先に進む前に原因を調べてください。

In [ ]:
reshaped["diff"].count(dims).rename("Valid HEALPix cells").plot()

In [ ]:
# 結果を保存します
reshaped.chunk(default_chunk_dims).to_zarr(
    f"{target_root}/diff-regridded.zarr",
    mode="w",
    consolidated=True,
    compute=True,
    storage_options=storage_options,
    zarr_format=2,
)

次の地図は、水温の差（`diff`）とモデルの海底の深さ（`H0`）を示しています。水温の差の地図では、0 に近い値を探してください。よく一致する場所は 1 か所に集まっていますか、それとも何か所かに分かれていますか？ このパターンが時間とともにどう変わるかも確認してください。

In [ ]:
reshaped.healpix.plot(var=["diff", "H0", "ocean_mask"], alpha=0.5)

## 4. 水温と合う場所を見つける

モデルの水温がタグの観測と一致する場所ほど、大きな重みを与えます。この重みを**観測確率（emission probability）** と呼びます（統計学では尤度（likelihood）にあたります）。観測が、それぞれの候補の場所とどのくらい整合するかを表します。

水温の差 0 を中心とするガウス分布（正規分布）の曲線を使います。設定 `differences_std` はその幅を決めます。値を大きくすると、より大きな水温の差まで許容します。放流と再捕（または死亡）の場所も、位置を絞り込む追加の手がかりになります。

`emission_pdf["pdf"]` に保存される水温の重みは、この段階ではまだ場所全体で合計しても 1 になりません。セクション 6 で手がかりを統合し、正規化します。

In [ ]:
from pangeo_fish.helpers import compute_emission_pdf

In [ ]:
# 前のステップのデータセットを開きます（ノートブックをここから再開するときだけ必要です）
differences = xr.open_dataset(
    f"{target_root}/diff-regridded.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

# ……そして観測確率の行列を計算します
emission_pdf = compute_emission_pdf(
    diff_ds=differences,
    events_ds=tag["tagging_events"].ds,
    differences_std=differences_std,
    initial_std=initial_std,
    recapture_std=recapture_std,
    dims=dims,
    chunk_time=chunk_time,
)[0]
emission_pdf

水温の重み（`pdf`）を、放流（`initial`）と最終位置（`final`、ある場合）の地図と比べてください。水温の観測から、狭い範囲に絞り込めていますか？ それとも、可能性のある海域がいくつも残っていますか？

In [ ]:
emission_pdf.healpix.plot(["initial", "final", "pdf"])

各時刻の水温による尤度の合計 `total = sum(dims, min_count=1)` を確認します。この段階では 1 になる必要はありません。

- `NaN`：その時刻にはデータがありません。
- `0`：データはありますが、タグの水温と一致する場所がひとつもありません。

どちらかが現れたら、先に進む前に原因を調べてください。

In [ ]:
emission_pdf = emission_pdf.chunk(default_chunk_dims).persist()
total = (
    emission_pdf["pdf"].sum(dims, min_count=1).rename("Total temperature likelihood")
)
print("Time steps without data (NaN):", int(total.isnull().sum()))
print("Time steps with zero total weight:", int((total == 0).sum()))
total.plot()

In [ ]:
# データセットを保存します
emission_pdf.to_zarr(
    f"{target_root}/emission_diff.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
emission_pdf.attrs["sources"] = emission_pdf.attrs.get("sources", []) + [
    "temperature"
]  # これまでに加えた手がかりの種類を記録します。
emission_pdf.to_zarr(
    f"{target_root}/total_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

## 5. 音響テレメトリーの受信記録を加える

受信記録は、その時刻に魚が受信機の想定受信範囲の中にいたことを教えてくれます。この情報を使って、水温による手がかりを補いながら、魚がいた可能性のある場所を絞り込みます。

このステップには、タグのデータに音響テレメトリーの受信記録と受信機の位置が含まれている必要があります。設定 `receiver_buffer` で想定する受信範囲を決めます。この手がかりは、セクション 6 で水温の手がかりと統合します。

In [ ]:
from pangeo_fish.helpers import compute_acoustic_pdf

In [ ]:
# 前のステップの観測確率を読み込み、音響テレメトリーの受信記録にもとづく観測確率を計算します
emission_diff = xr.open_dataset(
    f"{target_root}/emission_diff.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
acoustic_pdf = compute_acoustic_pdf(
    emission_ds=emission_diff,
    tag=tag,
    receiver_buffer=receiver_buffer,
    chunk_time=chunk_time,
    dims=dims,
)[0]
acoustic_pdf = acoustic_pdf.persist()
acoustic_pdf

In [ ]:
acoustic_pdf.healpix.plot("acoustic")

次のプロットは、受信の時刻と、セル数を数えた診断用の曲線を比べるものです。受信の時刻の前後で変化があるかを見てください。

ここでは、`acoustic != 0` を満たすセルを数えています。欠損値もこの条件を満たすため、この曲線は魚がいた可能性のある場所の正確な数ではありません。この点に注意して読み取ってください。

In [ ]:
tag["acoustic"]["deployment_id"].hvplot.scatter(c="red", marker="x") * (
    acoustic_pdf["acoustic"] != 0
).sum(dim=dims).hvplot()

受信があった時刻には、想定した受信範囲のもとで、可能性のある場所が受信機の周りに一気に絞り込まれることがあります。受信記録は、位置を強く制約する手がかりです。曲線の高さを可能性のある場所の正確な数と考えるのではなく、受信の時刻と合わせて曲線の変化を読み取ってください。

次のセルでは、必要に応じて音響テレメトリーの結果を別に保存します。このあとの解析では、すでに計算した結果をそのまま使えます。

In [ ]:
acoustic_pdf.to_zarr(
    f"{target_root}/acoustic.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

### 手がかりをまとめる

In [ ]:
# いま保存されている手がかりを開きます。
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
if "acoustic" in current_total.data_vars:
    pass
else:
    current_total = current_total.load()
    current_total.attrs["sources"] = current_total.attrs.get("sources", []) + [
        "acoustic"
    ]

    merged_pdf = current_total.merge(
        acoustic_pdf.drop_indexes(["cell_ids"]), compat="override"
    )

    merged_pdf.to_zarr(
        f"{target_root}/total_pdf.zarr",
        mode="w",
        consolidated=True,
        storage_options=storage_options,
        zarr_format=2,
    )
merged_pdf

## 6. 海底地形の情報を加える

このステップでは、GEBCO や EMODnet などの海底地形データを使って、もうひとつの観測確率の分布を計算します。

この確率を加えることで、前のステップで作った観測確率に海底地形の情報が取り込まれます。

**注意：** セクション 2 を少なくとも 1 回実行して結果を保存しておく必要があります。また、`relative_depth_threshold` は 0 にしておいてください。

### データの読み込み

海洋の参照モデルと、セクション 2 で計算した水温の差（diff）を読み込みます。海底地形のデータも読み込みます。

In [ ]:
if "reference_model" in locals():
    pass
else:
    from pangeo_fish.helpers import load_model

    reference_model = load_model(
        uri=ref_url,
        tag_log=tag_log,
        time_slice=time_slice,
        bbox=(bbox | {"max_depth": tag_log["pressure"].max()}),
        chunk_time=chunk_time,
        remote_options={},
        copernicus_model_name=copernicus_model_name,
    )
emission_diff = xr.open_dataset(
    f"{target_root}/emission_diff.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

In [ ]:
import fsspec
import numpy as np
from pangeo_fish.cf import bounds_to_bins
from pangeo_fish.tags import adapt_model_time, reshape_by_bins, to_time_slice
from pangeo_fish.bathy import batch_compute_pdf_bathy
from pangeo_fish.bathy import (
    compute_healpix_histogram_region_bin_size,
)

# bathy_url は海底地形ファイルのパスです
bathy_url = "https://data-taos.ifremer.fr/global_bathy/GEBCO_2024.zarr"

nside = (
    2**refinement_level
)  # refinement level に相当するパラメータ。一部の関数で使います
full_bathy = xr.open_dataset(
    "https://data-taos.ifremer.fr/demo/database/gebco_2024_new.zarr/",
    engine="zarr",
    chunks={},
    storage_options=None,
).rename({"lat": "latitude", "lon": "longitude"})
full_bathy

In [ ]:
subset_bathy = full_bathy.sel(
    {dim: slice(bounds[0] - 0.1, bounds[1] + 0.1) for dim, bounds in bbox.items()}
)

In [ ]:
ds_histo = compute_healpix_histogram_region_bin_size(
    subset_bathy,
    nside=nside,
    max_depth_m=1000,  # <- 最大深度（m）
    depth_bin_size=16,  # <- ビンの幅（m）
    ellipsoid=ellipsoid,
)

確率分布を計算します。

In [ ]:
from pangeo_fish.bathy import batch_compute_pdf_bathy

reshaped_tag = reshape_by_bins(  # 確率分布を計算するためにタグのデータを整形します
    tag_log,
    dim="time",
    bins=(
        reference_model.cf.add_bounds(["time"], output_dim="bounds")
        .pipe(bounds_to_bins, bounds_dim="bounds")
        .get("time_bins")
    ),
    other_dim="obs",
).chunk({"time": chunk_time})

pdf_da_func = batch_compute_pdf_bathy(
    ds_histo,
    reshaped_tag,
    target_root,
    batch_size=10000,
)

sum_over_cells = pdf_da_func.sum(dim="cells", skipna=True)
bathy_pdf = pdf_da_func / sum_over_cells  # 正規化
bathy_pdf

次のセルでは、必要に応じて海底地形の結果を別に保存します。このあとの解析では、すでに計算した結果をそのまま使えます。

In [ ]:
bathy_pdf.compute().to_zarr(
    f"{target_root}/bathy_pdf_{tag_name}.zarr",
    compute=True,
    mode="w",
    consolidated=True,
    zarr_format=2,
    storage_options=storage_options,
)

In [ ]:
# 確率分布を見たいとき
bathy_pdf.healpix.plot("pdf_bathy", refinement_level=refinement_level)

### 手がかりをまとめる

In [ ]:
## ほかの確率分布が入ったデータセットを開きます
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

In [ ]:
if "pdf_bathy" in current_total.data_vars:
    merged_pdf = current_total
    pass
else:
    current_total = current_total.load()
    current_total.attrs["sources"] = list(
        set(current_total.attrs.get("sources", [])) | {"bathymétrie"}
    )
    merged_pdf = current_total.merge(
        bathy_pdf.drop_indexes(["cell_ids"]), compat="override"
    )

    merged_pdf.to_zarr(
        f"{target_root}/total_pdf.zarr",
        mode="w",
        consolidated=True,
        storage_options=storage_options,
        zarr_format=2,
    )
merged_pdf

In [ ]:
liste = multiplot_healpix(
    [(merged_pdf, ["pdf", "pdf_bathy"])],
)

multi_map_with_synced_sliders(liste)

## 7. 潮汐による手がかりを加える

ノートブックの冒頭にある **解析の設定** で、**`tide_method`** を選びます。

- **`"legacy"`**（デフォルト）：TPXO の緯度・経度の格子上で 1 時間ごとの尤度を計算し、それを HEALPix に再格子化して、日ごとに平均します。
- **`"healpix"`**：セルごとの調和定数（仮）を使って、HMM のセル上で直接潮汐を予測します。魚が海底近くにいたと考えられる時間帯について、1 時間ごとの深度の平均と、予測した潮汐の平均を比べ、日ごとに未知の平均深度を推定します。これを選ぶ前に、オプションの `pangeo-fish[tide]` をインストールしてください。

どちらの方法もデフォルトでは TPXO9 を使い、深度にもとづく既存の潮汐と行動の判定フラグを共通で使います。FES2022b を使うには、`tide_model = "fes2022b"`、HEALPix の方法、`ellipsoid = "WGS84"` を選び、`tide_model_source` にローカルの AVISO の NetCDF ファイルを指定するか、`tide_constants_store` に計算済みの Zarr を指定します。出力を比べるときは、実行ごとに別の `scratch_root` を使ってください。モデルの仮定については、[潮汐の手法のドキュメント](https://github.com/pangeo-fish/pangeo-fish/blob/main/docs/tide.rst)を参照してください。

In [ ]:
import fsspec

from pangeo_fish.tide import (
    convert_lon_360_to_180,
    detect_tide,
    merge_tide_with_total,
    plot_tide,
    tide_pdf,
    tide_to_healpix,
)

In [ ]:
tag_test = detect_tide(tag_log, tag_name)
plot_tide(tag_test)
tag_test

### 選んだ方法で日ごとの潮汐の確率分布を作る

TPXO9 では、デモ用の URL またはローカルの TMD3 ファイルを指定できます。FES2022b では、AVISO の元の NetCDF ファイルか、計算済みの WGS84 の HEALPix Zarr を選びます。

In [ ]:
if not tide_constants_store:
    model_source = tide_model_source or (
        "https://data-taos.ifremer.fr/demo/database/Subset_NS_EC_forTMD30_21_2_2025.nc"
    )
    if tide_model == "fes2022b":
        model_source = str(Path(model_source).expanduser())
    tide_file = fsspec.open_local(
        f"simplecache::{model_source}"
        if model_source.startswith(("https://", "http://"))
        else model_source
    )
print(f"Tide method: {tide_method} | Tide model: {tide_model}")

In [ ]:
if tide_method == "legacy":
    with xr.open_dataset(tide_file) as source:
        legacy_pdf = tide_pdf(tag_test, convert_lon_360_to_180(source, bbox))
    legacy_pdf = legacy_pdf.assign_coords(
        longitude=((legacy_pdf.longitude + 180) % 360) - 180
    ).sortby("longitude")
    tide_to_healpix(target_root, refinement_level, ellipsoid, dataset=legacy_pdf)
else:
    from pangeo_fish.tide_healpix import (
        PHASE_CONVENTION,
        constants_to_healpix,
        fes2022b_constants_to_healpix,
        latlon_constants_from_tmd,
        open_harmonic_constants,
        still_tag_tide_likelihood,
        write_tide_pdf_healpix,
    )

    if tide_constants_store:
        tide_storage_options = None
        if tide_constants_store.startswith("s3://"):
            tide_storage_options = {
                "config_kwargs": {"s3": {"addressing_style": "path"}}
            }
            if tide_s3_endpoint:
                tide_storage_options["endpoint_url"] = tide_s3_endpoint
        with open_harmonic_constants(
            tide_constants_store,
            cell_ids=merged_pdf.cell_ids,
            constituents=tide_constituents,
            storage_options=tide_storage_options,
            ellipsoid="WGS84",
            require_all_cells=True,
        ) as stored:
            if stored.attrs.get("tide_model") != tide_model:
                raise ValueError(f"Expected harmonic constants from {tide_model}")
            for name in ("tide_real", "tide_imag"):
                attrs = stored[name].attrs
                if (
                    attrs.get("units") != "m"
                    or attrs.get("phase_convention") != PHASE_CONVENTION
                ):
                    raise ValueError(f"Unexpected units or phase convention for {name}")
            constants = stored.load()
    elif tide_model == "fes2022b":
        constants = fes2022b_constants_to_healpix(
            tide_file, merged_pdf.cell_ids, constituents=tide_constituents
        )
    else:
        constants = constants_to_healpix(
            latlon_constants_from_tmd(tide_file, bbox=bbox), merged_pdf.cell_ids
        )
    likelihood = still_tag_tide_likelihood(
        tag_test.depth.to_series(),
        constants,
        sigma=tide_sigma_m,
        valid=(tag_test.tide_found > 0).to_series(),
        min_samples=tide_min_samples,
    )
    daily_behaviour = tag_test.behav_found.resample(time="1D").max()
    write_tide_pdf_healpix(likelihood, target_root, behav_found=daily_behaviour)

In [ ]:
tide_daily = xr.open_dataset(
    f"{target_root}/tide_pdf_healpix.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
tide_daily.assign(pdf=tide_daily.pdf.clip(min=0)).healpix.plot("pdf")

### ほかの確率分布と統合する

In [ ]:
emission_with_tide, tide_mask, behav_mask = merge_tide_with_total(
    target_root, storage_options
)
# Clough-Tocher 法による再格子化では、値がわずかに 0 を下回ることがあります。確率分布は負になってはいけません。
# abs ではなく clip を使います。0 を下回った分は、日平均をとった時点ですでに平均に含まれているためです。
emission_with_tide["pdf_tide"] = emission_with_tide["pdf_tide"].clip(min=0)
emission_with_tide.attrs["tide_method"] = tide_method
emission_with_tide.attrs["tide_model"] = tide_model

In [ ]:
emission_with_tide.to_zarr(
    f"{target_root}/total_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
emission_with_tide

## 8. 確率分布を統合する

それぞれの確率密度関数を掛け合わせ、最終位置の情報があればそれも加えます。そのうえで、各時刻で合計が 1 になるように、セル全体の重みの合計で割ります。これを**正規化**と呼びます。

この結果は、観測から見て、それぞれの候補の場所がどのくらい支持されるかを表します。場所を時間方向につなぐには、さらに移動モデルが必要です。

In [ ]:
from pangeo_fish.helpers import normalize_pdf

In [ ]:
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
emission_pdf = normalize_pdf(
    ds=current_total,
    chunks=default_chunk_dims,
    dims=dims,
    exclude=("initial", "final", "mask"),
    excluded_pdf=None,  # ("pdf",)
)[0]

emission_pdf.to_zarr(
    f"{target_root}/combined.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
combined = emission_pdf

emission_pdf["pdf"].sum(dims).hvplot(
    width=500, ylim=(0, 2), title="Total probability per time step"
)

In [ ]:
variables_a_exclure = ["final", "initial", "mask"]

noms = [v for v in current_total.data_vars if v not in variables_a_exclure]
noms
liste = multiplot_healpix(
    [(current_total, noms)],
)

multi_map_with_synced_sliders(liste)

確率の合計は、どの時刻でもほぼ **1** になるはずです。データのない時刻も、1 からずれた値（合計が 0）として現れます。そのような時刻があれば、移動モデルを推定する前に原因を調べてください。

In [ ]:
combined.healpix.plot("pdf")

## 9. 魚がどのくらいの範囲を移動できるかを推定する

移動モデルは、連続する時刻のあいだで、魚がいた可能性のある場所どうしをつなぎます。そのパラメータ `sigma` は、移動がどのくらい広がりうるかを決めます。値が大きいほど、より広い範囲の移動を許します。このパラメータを観測から推定します。

次のセルを実行して、移動のパラメータを推定し、保存します。探索が警告なしに終わることを確認してください。表示される `f(x)` の値は、試した `sigma` の値での当てはまりの良さを表します。探索ではいろいろな値を試すので、毎回改善しなくても問題ありません。

移動の時間間隔は、データの日付から決まります。動画の設定 `time_step` を変えても、モデルの中の魚の移動は変わりません。

<details>
<summary>補足：パラメータの探索のしくみ</summary>

探索にはブレント法（Brent's method）を使い、`1e-4` ラジアンから上限 `maximum_speed × Δt × adjustment_factor / earth_radius` までの範囲で探します。ここで Δt は観測確率の時間座標の間隔、`earth_radius` は HEALPix 格子に保存された楕円体から求めた地球の半径（`"sphere"` の場合は約 6371 km）です。速度の設定は `sigma` の探索範囲を決めるもので、ありうる移動のひとつひとつを厳密に制限するものではありません。

推定された値が探索範囲の端に近くないかを確認してください。探索が収束しても、その解がただひとつであることや、全体での最適解であることが保証されるわけではありません。付録では、別の値と比べることができます。推定したパラメータと計算の設定は、次のステップのために JSON ファイルに保存されます。

</details>

In [ ]:
from pangeo_fish.helpers import optimize_pdf

### sigma を 1 つだけ最適化する

行動を考慮せずに、畳み込みの sigma を最適化します。「[Pangeo-Fish による魚類の位置推定](pangeo-fish.ipynb)」のノートブックを参照してください。

### 複数の sigma を最適化する（Multivariate optimisation）

この最適化では、行動の情報を使って sigma を時間によって変えます。こうすることで、検出された行動にもとづいて、魚の移動を日ごとに制約できます。

ここでは潮汐の情報を使います。

> **注意：** このデモの 2 つの `sigma` の値は、機能の使い方を示すためのもので、このタグに対する信頼できる推定値ではありません。日ごとのステップが 11 個しかなく、放流地点と最終地点が 3.6 km しか離れておらず、行動のグループごとの遷移も少ないため、データからはどちらの値も弱くしか決まりません。また、最適化は収束する前に 20 回の評価で打ち切られるため、返された値が最小値とは限りません（`conv_method="FFTConv"` では、スコアは `sigma` に対してなめらかに変化します）。推定された 2 つの値は慎重に解釈してください。

In [ ]:
fit_root = f"{target_root}/two_sigma"

from pangeo_fish.helpers import (
    compute_detection_time_intervals,
    stamp_parameter_indices,
    stamp_parameter_indices_from_mask,
)
import numpy as np
import matplotlib.pyplot as plt

emission = xr.open_dataset(
    f"{target_root}/combined.zarr",
    engine="zarr",
    chunks={},
    inline_array=True,
    storage_options=storage_options,
).chunk(default_chunk_dims)

with xr.open_dataset(
    f"{target_root}/tide_pdf_healpix.zarr",
    engine="zarr",
    storage_options=storage_options,
) as tide_daily:
    behav_mask = tide_daily.behav_found.sel(time=emission.time).values

behav_mask = (behav_mask > 0).astype(np.int8)
plt.plot(behav_mask)
pdf = stamp_parameter_indices_from_mask(
    pdf=emission,
    mask=behav_mask,
    sigma_bounds_dict={0: (0, maximum_speed), 1: (0, maximum_speed / 2)},
)

In [ ]:
from pangeo_fish.helpers import optimize_pdf

params, updated_emission = optimize_pdf(
    ds=pdf,
    adjustment_factor=adjustment_factor,
    truncate=truncate,
    maximum_speed=maximum_speed,
    tolerance=tolerance,
    dims=dims,
    single_behav=True,
    multivariate=True,
    # "two_sigma" という別のサブフォルダーを使います
    save_parameters=True,
    storage_options=storage_options,
    target_root=f"{fit_root}",
    conv_method="FFTConv",
)

updated_emission.attrs.pop("sigma_bounds", None)
updated_emission.to_zarr(
    f"{fit_root}/combined.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
# 描画では、推定したモデルと同じ場所にある格子のメタデータを読み込みます。
with open(f"{fit_root}/resampling_info.json", "w") as f:
    json.dump(resampling_info, f)
params

## 10. 位置を推定し、移動経路をまとめる

観測による手がかりと、推定した移動モデルを組み合わせて、各時刻にそれぞれの候補の場所に魚がいた確率を推定します。この確率分布を、次の 3 通りの方法でまとめます。

- **`viterbi_healpix`：** ビタビ（Viterbi）アルゴリズムで求めた、もっとも確からしい位置の系列全体。
- **`mode`：** 各時刻で確率がもっとも高いセル（最頻値）。
- **`mean`：** 各時刻の平均の位置。

これらの線は推定した確率分布の要約であり、実際に観測された経路ではありません。読み取るときは、確率が地図上でどのくらい広がっているかもあわせて確認してください。移動経路の連続する点のあいだの距離を km で、それに対応する速度を km/h で計算します。

<details>
<summary>補足：ほかの移動経路の求め方と計測値</summary>

ビタビ（Viterbi）アルゴリズムは、時刻ごとに別々に要約するのではなく、もっとも確からしい位置の系列全体を求めます。

`additional_track_quantities` の設定に指定できるのは `"speed"` と `"distance"` だけです。解析のあとで、返された移動経路から MovingPandas を使って、加速度（`add_acceleration`）、方向（`add_direction`）、方向の変化（`add_angular_difference`）、時間間隔（`add_timedelta`）を計算できます。これらは別の処理ステップです。[MovingPandas の trajectory API](https://movingpandas.readthedocs.io/en/main/api/trajectory.html) を参照してください。

</details>

In [ ]:
from pangeo_fish.helpers import compute_states

states = compute_states(
    target_root=f"{fit_root}",
    storage_options=storage_options,
    chunks=default_chunk_dims,
    save=True,
)

`states` の確率は、各時刻で合計がほぼ **1** になるはずです。1 からずれている場合は、その時刻に欠損値や不正な値があることを意味します。

In [ ]:
states["states"].sum(dims).hvplot(
    width=500, ylim=(0, 2), title="Total probability per time step"
)

In [ ]:
states = xr.open_dataset(
    f"{fit_root}/states.zarr",
    engine="zarr",
    chunks={},
    inline_array=True,
    storage_options=storage_options,
).chunk(default_chunk_dims)

In [ ]:
states.healpix.plot("states")

## 11. 地図と移動経路を見る

ビタビ（viterbi_healpix）、最頻値（mode）、平均（mean）の移動経路を比べ、確率の地図が時間とともにどう変わるかを見てみましょう。魚の位置がはっきり絞り込まれているのはどこでしょうか？ 可能性のある海域がいくつも残っているのはどこでしょうか？

**観測確率（emission）** の地図は、観測による手がかりを示します。**状態確率（state）** の地図は、時刻間の移動も考慮しています。両者のパターンを比べて、移動モデルが推定位置にどう影響しているかを確認してください。

### 11.1 移動経路を比べる

3 つの移動経路は、どこで一致し、どこで食い違っていますか？ 確率の地図を使って、その違いを読み解いてください。

In [ ]:
from pangeo_fish.helpers import decode_trajectories

trajectories = decode_trajectories(
    target_root=f"{fit_root}",
    storage_options=storage_options,
    chunks=default_chunk_dims,
    track_modes=track_modes,
    additional_track_quantities=additional_track_quantities,
    save=True,
)

In [ ]:
from pangeo_fish.helpers import plot_trajectories

plot_trajectories(
    target_root=f"{fit_root}",
    track_modes=track_modes,
    storage_options=storage_options,
    save_html=True,
)

### 11.2 確率分布の時間変化を見る

In [ ]:
from pangeo_fish.helpers import open_distributions, render_distributions

In [ ]:
data = open_distributions(
    target_root=f"{fit_root}",
    storage_options=storage_options,
    chunks=default_chunk_dims,
    chunk_time=chunk_time,
)
data

上の出力は、プロットに使えるデータの概要です。次のセルを実行すると、MP4 のアニメーションが作られ、コードのすぐ下で再生されます。

`time_step = 1` はデータのすべての時刻でフレームを作り、`3` は 3 時刻ごとにフレームを作ります。変わるのは動画のコマの間引きだけです。現在の動画作成ツールでは、この設定が 1 でも、最後の時刻は含まれません。

可能性のある場所が絞り込まれる時期や広がる時期に注目し、音響テレメトリーの受信の時刻と比べてみてください。

In [ ]:
import numpy as np

lon = data["longitude"].values  # (nx,)
lat = data["latitude"].values  # (ny,)
lon_2d, lat_2d = np.meshgrid(
    lon, lat
)  # (ny, nx)、2 次元の地理座標

data_2d = data.rename({"latitude": "y", "longitude": "x"})
data_2d = data_2d.assign_coords(
    longitude=(("y", "x"), lon_2d),
    latitude=(("y", "x"), lat_2d),
)
data_2d

In [ ]:
video_filename = render_distributions(
    data=data_2d,
    output_path=f"{fit_root}/states",
    xlim=bbox["longitude"],
    ylim=bbox["latitude"],
    time_step=time_step,
    extension="mp4",
    frames_dir="video",
    remove_frames=True,
    storage_options=storage_options,
)

In [ ]:
from IPython.display import Video, display

display(Video(filename=video_filename, embed=True))

In [ ]:
print("Your results are in:", target_root)

## 研究で使うときは

このノートブックの手法を自分の研究で使った場合は、pangeo-fish の論文（Magin et al., 2026, *Journal of Open Source Software*, [doi:10.21105/joss.08332](https://doi.org/10.21105/joss.08332)）を引用してください。一緒に引用するデータやツール、pangeo-fish を使った研究の一覧は「[研究で pangeo-fish を使ったら：引用のしかた](cite.md)」のページにまとめています。

## 補足の付録：移動の仮定を比べる

ワークショップの本編はここまでです。移動の仮定がどう影響するかを調べたい方のために、次のセルでは、推定したモデルと、指定した 2 つの `sigma` の値によるモデルを比べます。確率の地図と移動経路がどう変わるかを見てください。

比較ごとに別の出力フォルダーを使います。最初の比較では、用意された設定のまま使ってください。より発展的な解析では、期間ごとに異なる移動のパラメータを使うこともできます。

In [ ]:
import json
import warnings
from pathlib import Path
import fsspec
from pangeo_fish.helpers import create_parameters, hand_test_parameter
from pangeo_fish.helpers import optimize_pdf

まず、統合した観測確率の分布を別の出力フォルダーにコピーします。

In [ ]:
# 確率分布を開きます
emission = xr.open_dataset(
    f"{fit_root}/combined.zarr",
    engine="zarr",
    chunks={},
    inline_array=True,
    storage_options=storage_options,
).chunk(default_chunk_dims)
# 新しいフォルダーを作ります。元データのエンコーディングではなく、現在の Dask のチャンクを使います。
for name in emission.variables:
    emission[name].encoding.pop("chunks", None)
emission.to_zarr(
    f"{fit_root}/hand_sigma0.1/combined.zarr",  # "hand_sigma0.1" は別の名前に変えてもかまいません
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
emission

指定した移動のパラメータを、同じフォルダーに保存します。

In [ ]:
params = create_parameters(
    sigma=[0.0003],
    sigma_indices=None,
    n_steps=emission.sizes["time"],
    single_behav=True,
    class_name="FFTGaussian1DHealpix",  # 移動モデルの畳み込みクラス。
    target_root=f"{fit_root}/hand_sigma0.1",  # 上で使ったフォルダーと同じにします。
    save_parameters=True,
)
params

別の方法として、`hand_test_parameter()` を使うと、統合した観測確率の分布と指定したパラメータを比較用のフォルダーに保存できます。既存のデータセットを `emission` として渡すか、`None` のままにしてセクション 6 の `combined.zarr` を読み込みます。

In [ ]:
params = hand_test_parameter(
    emission=None,
    sigma_tested=[0.03],
    sigma_indices=None,
    Conv_method="FFTGaussian1DHealpix",
    target_root=fit_root,
    saving_root="/hand_sigma10",
    default_chunk_dims=default_chunk_dims,
    storage_options=storage_options,
)

In [ ]:
from pangeo_fish.helpers import compute_states

states_list = [
    compute_states(
        target_root=f"{fit_root}/{subfolder}",
        storage_options=storage_options,
        chunks=default_chunk_dims,
        save=True,
    )
    for subfolder in [
        "",
        "hand_sigma0.1",
        "hand_sigma10",
    ]
]

同期した地図で状態確率の分布を比べ、その下で移動経路を比べます。

In [ ]:
from pangeo_fish.helpers import multi_map_with_synced_sliders, multiplot_healpix

liste = multiplot_healpix(
    [
        (states_list[0], ["states"]),
        (states_list[1], ["states"]),
        (states_list[2], ["states"]),
    ],
)
liste

multi_map_with_synced_sliders(liste)

In [ ]:
from pangeo_fish.helpers import decode_trajectories

traj_list = [
    decode_trajectories(
        target_root=f"{fit_root}/{subfolder}",
        storage_options=storage_options,
        track_modes=comparison_track_modes,
        chunks=default_chunk_dims,
        save=True,
    )
    for subfolder in [
        "",
        "hand_sigma0.1",
        "hand_sigma10",
    ]
]

traj_plots = [
    plot_trajectories(
        target_root=f"{fit_root}/{subfolder}",
        track_modes=comparison_track_modes,
        storage_options=storage_options,
        save_html=True,
    ).options(title=f"Model's Folder: {subfolder}")
    for subfolder in [
        "",
        "hand_sigma0.1",
        "hand_sigma10",
    ]
]
(traj_plots[0] + traj_plots[1] + traj_plots[2]).cols(2)

## 付録：バグ報告用の情報（Information for bug reports）

うまく動かないときは、どのソフトウェアを使っていたかを正確に知る必要があります。次のセルを実行し（前のセルでエラーが出ていても、いつでも実行できます）、**出力をすべて**コピーして、エラーメッセージと一緒に [issue](https://github.com/pangeo-fish/pangeo-fish-fra2026/issues) やメールに貼り付けてください。

出力からは、次の 3 つのことがわかります。

- **どの計算環境か：** Jupyter サーバーの起動に使った Docker イメージ（レジストリ、名前、タグ。例：`ghcr.io/pangeo-fish/pangeo-fish-notebook:2026.10.21-8fb0870`）。タグの末尾は、イメージの作成に使った Pangeo-Fish のコミットです。
- **どの Pangeo-Fish のコードか：** そのイメージにインストールされている Pangeo-Fish のソースの Git リポジトリとコミット。
- **どのノートブックか：** このワークショップのリポジトリのコミットと、ノートブックを変更したかどうか。

In [ ]:
# バグ報告用のソフトウェア情報：出力をすべてコピーして報告に貼り付けてください。
import os
import platform
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version


def run(*cmd):
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except Exception as err:
        return f"unavailable ({type(err).__name__})"


def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"


git = shutil.which("git", path=os.path.join(sys.prefix, "bin")) or "git"
fish_src = os.environ.get("PANGEO_FISH_SRC", "/srv/pangeo-fish")

print(
    "Docker image       :",
    os.environ.get("JUPYTER_IMAGE", "unknown (not set by the hub)"),
)
print(
    "JupyterHub         :",
    os.environ.get(
        "JUPYTERHUB_SERVICE_URL", os.environ.get("JUPYTERHUB_API_URL", "unknown")
    ),
)
print("Pangeo-Fish version:", pkg_version("pangeo-fish"))
print("Pangeo-Fish source :", run(git, "-C", fish_src, "remote", "get-url", "origin"))
print(
    "Pangeo-Fish commit :",
    run(git, "-C", fish_src, "log", "-1", "--format=%H (%cs) %s"),
)
print("Notebook repository:", run(git, "remote", "get-url", "origin"))
print("Notebook commit    :", run(git, "log", "-1", "--format=%H (%cs) %s"))
print("Notebook changes   :", run(git, "status", "--short", ".") or "none")
print("Python             :", sys.version.split()[0], "on", platform.platform())
for name in ["xarray", "dask", "numpy", "xdggs", "healpix-geo", "copernicusmarine"]:
    print(f"{name:19}:", pkg_version(name))